# Text Feature Extraction — MELD

Encoder: configurable via `MODEL_NAME` in the config cell (default: `roberta-large`)  
Feature: CLS token from last hidden state → shape `(HIDDEN_SIZE,)` per utterance  
Output: `Dataset/Processed/MELD/features/text_{MODEL_TAG}_{split}.pt`  
Format: `dict {clip_name: np.array(HIDDEN_SIZE,)}` per split  

Expected counts: train=9989, dev=1109, test=2610

In [1]:
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from transformers import AutoTokenizer, AutoModel
from tqdm.notebook import tqdm

In [ ]:
DATA_ROOT    = Path("Dataset/Processed/MELD")
FEATURES_DIR = DATA_ROOT / "features"
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 8
MAX_LEN    = 512
SPLITS     = ["train", "dev", "test"]

# ── Swap model here ───────────────────────────────────────────────────────────
MODEL_NAME = "roberta-large"          # any HuggingFace AutoModel-compatible ID
# Examples:
#   "roberta-large"                      → text_roberta_large_{split}.pt        (hidden=1024)
#   "bert-base-uncased"                  → text_bert_base_uncased_{split}.pt    (hidden=768)
#   "bert-large-uncased"                 → text_bert_large_uncased_{split}.pt   (hidden=1024)
#   "microsoft/deberta-v3-large"         → text_microsoft_deberta_v3_large_{split}.pt (hidden=1024)
#   "google/electra-large-discriminator" → text_google_electra_large_discriminator_{split}.pt
# ─────────────────────────────────────────────────────────────────────────────
MODEL_TAG  = MODEL_NAME.replace("/", "_").replace("-", "_")  # safe filename slug
print(f"Model    : {MODEL_NAME}")
print(f"Tag      : {MODEL_TAG}")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device   : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM free: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB")

Model    : roberta-large
Tag      : roberta_large
Device   : cuda
GPU      : NVIDIA GeForce RTX 3060
VRAM free: 11.8 GB


In [3]:
labels = pd.read_csv(DATA_ROOT / "labels.csv")
print(f"Total utterances : {len(labels)}")
print(f"Columns          : {labels.columns.tolist()}")
print(f"Split counts     :")
print(labels["split"].value_counts())
labels.head(3)

Total utterances : 13708
Columns          : ['clip_name', 'split', 'dia_id', 'utt_id', 'utterance', 'speaker', 'emotion', 'sentiment', 'season', 'episode', 'start_time', 'end_time', 'audio_path', 'video_path', 'text_path', 'status']
Split counts     :
split
train    9989
test     2610
dev      1109
Name: count, dtype: int64


,clip_name,split,dia_id,utt_id,utterance,speaker,emotion,sentiment,season,episode,start_time,end_time,audio_path,video_path,text_path,status
0,dia0_utt0,train,0,0,also I was the point person on my companys tr...,Chandler,neutral,neutral,8,21,"00:16:16,059","00:16:21,731",audio/train/dia0_utt0.wav,video/train/dia0_utt0.mp4,text/train/dia0_utt0.txt,ok
1,dia0_utt1,train,0,1,You mustve had your hands full.,The Interviewer,neutral,neutral,8,21,"00:16:21,940","00:16:23,442",audio/train/dia0_utt1.wav,video/train/dia0_utt1.mp4,text/train/dia0_utt1.txt,ok
2,dia0_utt2,train,0,2,That I did. That I did.,Chandler,neutral,neutral,8,21,"00:16:23,442","00:16:26,389",audio/train/dia0_utt2.wav,video/train/dia0_utt2.mp4,text/train/dia0_utt2.txt,ok


In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModel.from_pretrained(MODEL_NAME)
model.eval()
for p in model.parameters():
    p.requires_grad_(False)
model = model.to(DEVICE)
HIDDEN_SIZE = model.config.hidden_size
print(f"Hidden size : {HIDDEN_SIZE}")
print(f"Params      : {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M")

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Hidden size : 1024
Params      : 355M


In [5]:
def extract_batch(texts: list[str]) -> np.ndarray:
    """Returns CLS-token embeddings, shape (B, HIDDEN_SIZE)."""
    enc = tokenizer(
        texts,
        max_length=MAX_LEN,
        truncation=True,
        padding=True,
        return_tensors="pt",
    )
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    with torch.no_grad():
        out = model(**enc)
    return out.last_hidden_state[:, 0, :].cpu().numpy()

In [6]:
EXPECTED_COUNTS = {"train": 9989, "dev": 1109, "test": 2610}

split_features: dict[str, dict[str, np.ndarray]] = {}

for split in SPLITS:
    split_df   = labels[labels["split"] == split].reset_index(drop=True)
    print(f"\n--- {split} ({len(split_df)} utterances) ---")

    features: dict[str, np.ndarray] = {}
    clip_names = split_df["clip_name"].tolist()
    text_paths = split_df["text_path"].tolist()

    batch_ids: list[str]   = []
    batch_texts: list[str] = []

    for i, (cname, tpath) in enumerate(
        tqdm(zip(clip_names, text_paths), total=len(clip_names), desc=split)
    ):
        txt_file = Path(tpath)
        text = txt_file.read_text(encoding="utf-8").strip() if txt_file.exists() else ""
        batch_ids.append(cname)
        batch_texts.append(text if text else "[empty]")

        if len(batch_texts) == BATCH_SIZE or i == len(clip_names) - 1:
            feats = extract_batch(batch_texts)
            for bid, feat in zip(batch_ids, feats):
                features[bid] = feat
            batch_ids.clear()
            batch_texts.clear()

    out_path = FEATURES_DIR / f"text_{MODEL_TAG}_{split}.pt"
    torch.save(features, out_path)
    size_mb = out_path.stat().st_size / 1e6
    print(f"Saved  : {out_path}  ({size_mb:.1f} MB, {len(features)} entries)")
    split_features[split] = features


--- train (9989 utterances) ---


train:   0%|          | 0/9989 [00:00<?, ?it/s]

Saved  : Dataset/Processed/MELD/features/text_roberta_large_train.pt  (62.2 MB, 9989 entries)

--- dev (1109 utterances) ---


dev:   0%|          | 0/1109 [00:00<?, ?it/s]

Saved  : Dataset/Processed/MELD/features/text_roberta_large_dev.pt  (6.9 MB, 1109 entries)

--- test (2610 utterances) ---


test:   0%|          | 0/2610 [00:00<?, ?it/s]

Saved  : Dataset/Processed/MELD/features/text_roberta_large_test.pt  (16.2 MB, 2610 entries)


In [7]:
print("=== Verification ===")

for split in SPLITS:
    out_path = FEATURES_DIR / f"text_roberta_large_{split}.pt"
    loaded   = torch.load(out_path, weights_only=False)

    expected = EXPECTED_COUNTS[split]
    assert len(loaded) == expected, f"{split}: Expected {expected}, got {len(loaded)}"

    sample_key  = next(iter(loaded))
    sample_feat = loaded[sample_key]
    assert sample_feat.shape == (1024,), f"{split}: shape {sample_feat.shape}"

    all_feats = np.stack(list(loaded.values()))
    has_nan   = np.isnan(all_feats).any()
    has_inf   = np.isinf(all_feats).any()
    assert not has_nan, f"{split}: NaN detected"
    assert not has_inf, f"{split}: Inf detected"

    l2_norm = np.linalg.norm(sample_feat)
    print(f"\n[{split}]")
    print(f"  Count   : {len(loaded)} / {expected}  ✓")
    print(f"  Shape   : {sample_feat.shape}  ✓")
    print(f"  Mean    : {all_feats.mean():.4f}")
    print(f"  Std     : {all_feats.std():.4f}")
    print(f"  Has NaN : {has_nan}  ✓")
    print(f"  Has Inf : {has_inf}  ✓")
    print(f"  Sample  : {sample_key}  |  L2 norm = {l2_norm:.2f}")

    split_row  = labels[labels.clip_name == sample_key].iloc[0]
    sample_txt = (DATA_ROOT / split_row.text_path).read_text(encoding="utf-8").strip()
    print(f"  Text    : {sample_txt[:100]}")
    print(f"  Emotion : {split_row.emotion}")

print("\nAll assertions passed.")

=== Verification ===

[train]
  Count   : 9989 / 9989  ✓
  Shape   : (1024,)  ✓
  Mean    : -0.0315
  Std     : 0.9893
  Has NaN : False  ✓
  Has Inf : False  ✓
  Sample  : dia0_utt0  |  L2 norm = 31.67
  Text    : also I was the point person on my companys transition from the KL-5 to GR-6 system.
  Emotion : neutral

[dev]
  Count   : 1109 / 1109  ✓
  Shape   : (1024,)  ✓
  Mean    : -0.0315
  Std     : 0.9893
  Has NaN : False  ✓
  Has Inf : False  ✓
  Sample  : dia0_utt0  |  L2 norm = 31.67
  Text    : also I was the point person on my companys transition from the KL-5 to GR-6 system.
  Emotion : neutral

[test]
  Count   : 2610 / 2610  ✓
  Shape   : (1024,)  ✓
  Mean    : -0.0315
  Std     : 0.9893
  Has NaN : False  ✓
  Has Inf : False  ✓
  Sample  : dia0_utt0  |  L2 norm = 31.67
  Text    : also I was the point person on my companys transition from the KL-5 to GR-6 system.
  Emotion : neutral

All assertions passed.
